In [16]:
import pyexasol
import configparser
import pandas as pd
import datetime
import os
from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import smtplib
import boto3
from multiprocessing.pool import ThreadPool
from unidecode import unidecode
import glob
%matplotlib inline

def connect(path):
    env = path.split('\\')[-1].split('.')[0]
    #Location of the ini file
    config = configparser.ConfigParser()
    config.read(path)

    dsn=config[env]['dsn']
    user=config[env]['user']
    pwd=config[env]['pwd']
    schema=config[env]['schema']

    # Exasol connection
    connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema,connection_timeout= 1000)
    return connect


def execute(config, query):
    c= connect(config)
    QUERY = c.execute(sql_query)
    # Importing data into a DataFrame
    df = c.export_to_pandas(sql_query)
    df['LDTS'] = datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    return df


def get_parameter(parameter_list: list) -> dict:
    """ This function reads a secure parameter from AWS' SSM service.

    :param parameter_list: list of valid parameter name(s) to be fetched from AWS SSM.
    :return: parameter's value
    """
    ssm = boto3.client('ssm', region_name='eu-central-1', verify=False)
    with ThreadPool(8) as pool:
        values = pool.map(lambda x: ssm.get_parameters(Names=[x], WithDecryption=True)['Parameters'][0]['Value'], parameter_list)
    return {parameter_list[i]: values[i] for i in range(len(parameter_list))}


def send_email(recipient):
    """ This function sends email with Geosure attachments. """
#     recipients_list = [f'{recipient}@hrs.com' for recipient in recipients]
    
    param_store = get_parameter([
                             "/hpo/email-smtp-host",
                             "/hpo/email-port",
                             "/hpo/email-user",
                             "/hpo/email-password"
                             ])
    
    settings = {"smtp": param_store['/hpo/email-smtp-host'],
                "port": param_store['/hpo/email-port'],
                "user": param_store['/hpo/email-user'],
                "password": param_store[f'/hpo/email-password']}
    message = MIMEMultipart()
    message["From"] = f"{param_store['/hpo/email-user']}@hrs.com"
    message["To"] = f"{recipient}@hrs.com"#"; ".join(recipients_list)
    print(message["To"])
    message["Subject"] = 'GEOSURE ALERT'
    body = """ THIS IS AN AUTOMATED EMAIL. PLEASE DO NOT REPLY
    Hi Procurement team,
    
    Please find the attached list of cities which have new Geosure scores. Kindly verify.
    
    Regards,
    Py"""
    
    message.attach(MIMEText(body, 'plain'))
    attachments = [f'Geosure_deviated_cities_{datetime.date.today().strftime("%d-%m-%Y")}.xlsx']

    for file in attachments:
        path = os.path.join(f'{os.getcwd()}', 'Geosure', file)
        part = MIMEBase("application", "octet-stream")
        part.set_payload(open(path, "rb").read())
        encoders.encode_base64(part)
        part.add_header("Content-Disposition", f"attachment; filename={unidecode(file)}")
        message.attach(part)

    s = smtplib.SMTP(host=settings['smtp'], port=settings['port'])
    s.sendmail(message["From"], message["To"], message.as_string())       
    s.quit()

In [17]:
sql_query="""SELECT 
                    GS_DISTRICT_NAME,
                    GS_CITY_NAME,
                    GS_COUNTRY_NAME, 
                    AVG(GS_COMPOSITE_SCORE) AS AVG_COMPOSITE_SCORE,
                    AVG(NIGHT_SAFE_SCORE) AS AVG_NIGHT_SAFE_SCORE,
                    AVG(PHYSICAL_SAFE_SCORE) AS AVG_PHYSICAL_SAFE_SCORE,
                    AVG(WOMEN_SAFE_SCORE) AS AVG_WOMEN_SAFE_SCORE,
                    AVG(THEFT_SCORE) AS AVG_THEFT_SCORE,
                    AVG(BASIC_FREEDOM_SCORE) AS AVG_BASIC_FREEDOM_SCORE,
                    AVG(HEALTH_MEDS_SCORE) AS AVG_HEALTH_MEDS_SCORE,
                    AVG(LGBTQ_SAFE_SCORE) AS AVG_LGBTQ_SAFE_SCORE
                FROM DWHPFX.GEOSURE_HOTELS 
                GROUP BY 
                    GS_DISTRICT_NAME,
                    GS_CITY_NAME,
                    GS_COUNTRY_NAME
                ORDER BY
                    GS_COUNTRY_NAME,
                    GS_CITY_NAME,
                    GS_DISTRICT_NAME"""

config_path='C:\\Users\\USER\\.spyder-py3\\pfxPROD.ini'
today = datetime.date.today()
yesterday = today - datetime.timedelta(days=-1)
wdir = os.path.join(f'{os.getcwd()}', 'Geosure')
file_type = '\*.xlsx'
files = glob.glob(wdir + file_type)
max_file = max(files, key=os.path.getctime)

df_old=pd.read_excel(max_file)
df_new = execute(config_path, sql_query)

In [19]:
## Check 1
if df_old.shape == df_new.shape:
    print('Success')
else:
    print('Failed')
    
## Check 2
final_df = df_old.merge(df_new, left_index=True, right_index=True,suffixes=('_OLD', '_NEW'))
column_values = final_df[['LDTS_OLD','LDTS_NEW']].values.ravel()
unique_values =  pd.unique(column_values)
print(unique_values)

In [20]:
## Check 3
final_df['CompositeChange'] = final_df['AVG_COMPOSITE_SCORE_OLD'] - final_df['AVG_COMPOSITE_SCORE_NEW']
final_df['NightSafeChange'] = final_df['AVG_NIGHT_SAFE_SCORE_OLD'] - final_df['AVG_NIGHT_SAFE_SCORE_NEW']
final_df['PhysicalSafeChange'] = final_df['AVG_PHYSICAL_SAFE_SCORE_OLD'] - final_df['AVG_PHYSICAL_SAFE_SCORE_NEW']
final_df['WomenSafeChange'] = final_df['AVG_WOMEN_SAFE_SCORE_OLD'] - final_df['AVG_WOMEN_SAFE_SCORE_NEW']
final_df['TheftChange'] = final_df['AVG_THEFT_SCORE_OLD'] - final_df['AVG_THEFT_SCORE_NEW']
final_df['FreedomChange'] = final_df['AVG_BASIC_FREEDOM_SCORE_OLD'] - final_df['AVG_BASIC_FREEDOM_SCORE_NEW']
final_df['HealthChange'] = final_df['AVG_HEALTH_MEDS_SCORE_OLD'] - final_df['AVG_HEALTH_MEDS_SCORE_NEW']
final_df['LGBTQChange'] = final_df['AVG_LGBTQ_SAFE_SCORE_OLD'] - final_df['AVG_LGBTQ_SAFE_SCORE_NEW']

ana_df=final_df[['GS_DISTRICT_NAME_OLD', 'GS_CITY_NAME_OLD', 'GS_COUNTRY_NAME_OLD',
                 'GS_DISTRICT_NAME_NEW', 'GS_CITY_NAME_NEW', 'GS_COUNTRY_NAME_NEW',
                 'CompositeChange', 'NightSafeChange', 'PhysicalSafeChange','WomenSafeChange', 
                 'TheftChange', 'FreedomChange', 'HealthChange', 'LGBTQChange']] 

df = ana_df.query('CompositeChange!=0 | NightSafeChange!=0 | PhysicalSafeChange!=0 \
             | WomenSafeChange!=0 | TheftChange!=0 | FreedomChange!=0 | HealthChange!=0 | LGBTQChange!=0')
# df = ana_df[(ana_df['CompositeChange'] == 0) & (ana_df['GS_CITY_NAME_OLD'].str.contains("Dusseldorf"))]
if ana_df.query('CompositeChange!=0 | NightSafeChange!=0 | PhysicalSafeChange!=0 \
                | WomenSafeChange!=0 | TheftChange!=0 | FreedomChange!=0 \
                | HealthChange!=0 | LGBTQChange!=0').shape[0] != 0:
        
    file = f'Geosure_deviated_cities_{today.strftime("%d-%m-%Y")}.xlsx'
    filepath = os.path.join(f'{os.getcwd()}', 'Geosure', file)
    df.to_excel(filepath, sheet_name='Geosure_deviation', index = False)
    
    recipients=['gbr03','vva02', 'asa07','svi02']
    for recipient in recipients:
        send_email(recipient)
        print(f"Email sent to the recipient: {recipient}")
else:
    print("All is well! I don't wish to spam you.")

oldDataFile = f'Geosure_data_{today.strftime("%d-%m-%Y")}.xlsx'
df_new.to_excel(os.path.join(wdir,oldDataFile), sheet_name=today.strftime("%d-%m-%Y"), index = False)

In [196]:
ana_df[(ana_df['CompositeChange'] == 0) & (ana_df['GS_CITY_NAME_OLD'].str.contains("Duss"))]

In [88]:
final_df[final_df['GS_CITY_NAME_OLD'].str.contains("Koln")] \
        [['GS_DISTRICT_NAME_OLD','AVG_COMPOSITE_SCORE_OLD', 'AVG_COMPOSITE_SCORE_NEW']] \
        .sort_values(['AVG_COMPOSITE_SCORE_OLD', 'AVG_COMPOSITE_SCORE_NEW'], axis=0, ascending=False) \
        .head(10).plot.bar(x='GS_DISTRICT_NAME_OLD')

In [98]:
final_df[final_df['GS_CITY_NAME_OLD'].str.contains("Koln")] \
        [['GS_DISTRICT_NAME_OLD','AVG_NIGHT_SAFE_SCORE_OLD','AVG_PHYSICAL_SAFE_SCORE_OLD', 
          'AVG_WOMEN_SAFE_SCORE_OLD', 'AVG_THEFT_SCORE_OLD', 'AVG_BASIC_FREEDOM_SCORE_OLD',
          'AVG_HEALTH_MEDS_SCORE_OLD', 'AVG_LGBTQ_SAFE_SCORE_OLD','AVG_COMPOSITE_SCORE_OLD']] \
        .sort_values(['AVG_COMPOSITE_SCORE_OLD'], axis=0, ascending=False) \
        .plot.line(x='GS_DISTRICT_NAME_OLD', figsize=(20, 10))

In [118]:
import numpy as np
import seaborn as sns
import matplotlib.pylab as plt

# uniform_data = np.random.rand(10, 12)
ax = sns.heatmap(final_df[final_df['GS_CITY_NAME_OLD'].str.contains("Koln")] \
        [['AVG_NIGHT_SAFE_SCORE_OLD','AVG_PHYSICAL_SAFE_SCORE_OLD', 
          'AVG_WOMEN_SAFE_SCORE_OLD', 'AVG_THEFT_SCORE_OLD', 'AVG_BASIC_FREEDOM_SCORE_OLD',
          'AVG_HEALTH_MEDS_SCORE_OLD', 'AVG_LGBTQ_SAFE_SCORE_OLD','AVG_COMPOSITE_SCORE_OLD']], figsize=(20, 10))
plt.show()